In [ ]:
# import knihoven
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from uncertainties import ufloat as uf
from scipy.optimize import curve_fit
from uncertainties import correlated_values
from pathlib import Path

# path config
downloads_dir = Path.home() / 'Downloads'
# ujistime se, ze slozka existuje (kdyby nahodou ne)
downloads_dir.mkdir(parents=True, exist_ok=True)
# output_path = downloads_dir / 'muj_graf.png'

In [ ]:
# definice funkcí

# nejistota typu A 
def unc_A(data):
    """
    Výpočet nejistoty typu A
    data : list nebo numpy array
         měření
    return : float
        nejistota typu A
    """
    data = np.array(data)
    n = len(data)

    s = np.std(data, ddof=1)
    u_A = s / np.sqrt(n)
    
    return u_A

# nacteni z df do jednotlivych arrays
def df_to_arrays(df, sloupce, od=0, do=None, dtype=float):
    if isinstance(sloupce, str):
        sloupce = [sloupce]

    chybejici = [s for s in sloupce if s not in df.columns]
    if chybejici:
        raise KeyError(f"Chybejici sloupce: {chybejici}")

    return [df[s].iloc[od:do].dropna().to_numpy(dtype=dtype) for s in sloupce]

# ufloat z aritmetickeho prumeru
def ufloat_unc_A(array):
    return uf(np.mean(array), unc_A(array))

# prepocet z popt a pcov na ufloaty
def popt_pcov_to_ufloat(popt, pcov, correlated=False):
    """
    Převede výsledek curve_fit (popt, pcov) na ufloaty.

    Parametry
    ---------
    popt : array_like   optimální parametry z curve_fit
    pcov : array_like   kovarianční matice z curve_fit
    correlated : bool   True  -> ufloaty nesou korelace mezi parametry
                        False -> nezávislé ufloaty (jen sqrt diagonály pcov)

    Vrací
    -----
    tuple[ufloat, ...]  parametry s nejistotami
    """
    popt = np.asarray(popt, dtype=float)
    pcov = np.asarray(pcov, dtype=float)

    if correlated:
        return tuple(correlated_values(popt, pcov))

    perr = np.sqrt(np.diag(pcov))
    return tuple(uf(v, e) for v, e in zip(popt, perr))

# array do latex
def array_to_latex(*array, dig=1, format='u', style='L', only_return=False):
    output = [f'${u:.{dig}{format}{style}}$' for u in array]
    if only_return:
        return output
    else:
        for s in output:
            print(s)

In [ ]:
# nacteni google tabulek
sheet_id = "1cXKdaqbpzERg_syz5NUfd-MyWxmNyKs_Tjba6bure2k"
gid = "2139038384"
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"
df_peaks = pd.read_csv(url)